# **__Binance Trading Accounts Analysis__**


### Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from ast import literal_eval

### Load the dataset

In [2]:
# Since 'Trade_History' is stored as a stringified list, we need to convert it back to a list format
df = pd.read_csv('TRADES_CopyTr_90D_ROI.csv')
df['Trade_History'] = df['Trade_History'].fillna('[]').apply(literal_eval) 

In [3]:
df.info()
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Port_IDs       150 non-null    int64 
 1   Trade_History  150 non-null    object
dtypes: int64(1), object(1)
memory usage: 2.5+ KB


,Port_IDs,Trade_History
0,3925368433214965504,"[{'time': 1718899656000, 'symbol': 'SOLUSDT', ..."
1,4002413037164645377,"[{'time': 1718980078000, 'symbol': 'NEARUSDT',..."
2,3923766029921022977,"[{'time': 1718677164000, 'symbol': 'ETHUSDT', ..."
3,3994879592543698688,"[{'time': 1718678214000, 'symbol': 'ETHUSDT', ..."
4,3926423286576838657,"[{'time': 1718979615000, 'symbol': 'ETHUSDT', ..."
5,3987739404272887297,"[{'time': 1718979652000, 'symbol': 'ONDOUSDT',..."
6,4030395639953224449,"[{'time': 1718981481000, 'symbol': 'ETHUSDT', ..."
7,3953433416230728705,"[{'time': 1718942316000, 'symbol': 'BTCUSDT', ..."
8,3919174299855478272,"[{'time': 1718853879000, 'symbol': 'ETHUSDT', ..."
9,4029749871687083265,"[{'time': 1718675040000, 'symbol': 'FXSUSDT', ..."


### Some accounts have multiple trades within the same record, so we need to "explode" them


In [4]:
df = df.explode('Trade_History')


### Convert nested dictionary structure into a proper dataframe


In [5]:
trades_df = pd.json_normalize(df['Trade_History'])

### Attach the corresponding account ID to each trade for tracking


In [6]:
trades_df = pd.concat([df[['Port_IDs']].reset_index(drop=True), trades_df], axis=1)


### Convert timestamps into readable dates

In [7]:
# Some timestamps might be invalid, so we set errors='coerce' to avoid crashes
trades_df['time'] = pd.to_datetime(trades_df['time'], errors='coerce', unit='ms')

### If any rows have NaT (missing timestamps), we drop them to ensure consistency

In [8]:
trades_df = trades_df.dropna(subset=['time'])

### Define a mapping to classify trades as long/short open/close based on 'side' and 'positionSide'


In [9]:

trade_type_map = {
    ('BUY', 'LONG'): 'long_open',
    ('SELL', 'LONG'): 'long_close',
    ('SELL', 'SHORT'): 'short_open',
    ('BUY', 'SHORT'): 'short_close'
}


### Apply the mapping function to each row


In [10]:
trades_df['trade_type'] = trades_df.apply(
    lambda row: trade_type_map.get((row['side'], row['positionSide']), 'unknown'), axis=1
)


## Function to compute financial metrics per account


In [11]:
def calculate_metrics(group):
    """Calculates key performance metrics for each trading account."""
    if group.empty:
        return pd.Series({
            'PnL': 0, 'Win_Positions': 0, 'Total_Positions': 0, 'Win_Rate': 0,
            'ROI': 0, 'Sharpe_Ratio': 0, 'MDD': 0
        })
    
    # Total profit/loss for the account
    pnl = group['realizedProfit'].sum()
    
    # Number of trades that resulted in a profit
    win_positions = (group['realizedProfit'] > 0).sum()
    
    # Total number of trades executed
    total_positions = len(group)
    
    # Win rate as a percentage
    win_rate = win_positions / total_positions if total_positions else 0
    
    # ROI (Return on Investment) - calculated as total profit divided by total investment
    investment = group.loc[group['trade_type'].isin(['long_open', 'short_open']), 'quantity'].sum()
    roi = pnl / investment if investment != 0 else 0  # Avoid division by zero
    
    # Sharpe Ratio: Measures risk-adjusted returns (higher is better)
    daily_pnl = group.set_index('time').resample('D')['realizedProfit'].sum()
    sharpe = (daily_pnl.mean() / daily_pnl.std()) * np.sqrt(252) if daily_pnl.std() != 0 else 0  # Annualized
    
    # Maximum Drawdown: Measures the worst peak-to-trough loss during the period
    cumulative_pnl = group['realizedProfit'].cumsum()
    mdd = (cumulative_pnl - cumulative_pnl.cummax()).min()
    
    return pd.Series({
        'PnL': pnl, 'Win_Positions': win_positions, 'Total_Positions': total_positions,
        'Win_Rate': win_rate, 'ROI': roi, 'Sharpe_Ratio': sharpe, 'MDD': mdd
    })


### Apply metric calculations per account

In [12]:
metrics_df = trades_df.groupby('Port_IDs', group_keys=False).apply(
    calculate_metrics, include_groups=False
).reset_index()

### Normalize metrics for fair ranking (all values between 0 and 1)


In [13]:
metrics_df['ROI_norm'] = (metrics_df['ROI'] - metrics_df['ROI'].min()) / (metrics_df['ROI'].max() - metrics_df['ROI'].min() + 1e-9)
metrics_df['Sharpe_Ratio_norm'] = (metrics_df['Sharpe_Ratio'] - metrics_df['Sharpe_Ratio'].min()) / (metrics_df['Sharpe_Ratio'].max() - metrics_df['Sharpe_Ratio'].min() + 1e-9)
metrics_df['Win_Rate_norm'] = (metrics_df['Win_Rate'] - metrics_df['Win_Rate'].min()) / (metrics_df['Win_Rate'].max() - metrics_df['Win_Rate'].min() + 1e-9)

### Since MDD (Maximum Drawdown) is a negative metric (lower is better), we invert the normalization


In [14]:
metrics_df['MDD_norm'] = 1 - (
    (metrics_df['MDD'] - metrics_df['MDD'].min()) / (metrics_df['MDD'].max() - metrics_df['MDD'].min() + 1e-9)
)

### Composite score to rank accounts (higher score = better performance)

In [15]:
weights = {'ROI': 0.3, 'Sharpe_Ratio': 0.3, 'Win_Rate': 0.2, 'MDD': 0.2}
metrics_df['Composite_Score'] = (
    weights['ROI'] * metrics_df['ROI_norm'] +
    weights['Sharpe_Ratio'] * metrics_df['Sharpe_Ratio_norm'] +
    weights['Win_Rate'] * metrics_df['Win_Rate_norm'] +
    weights['MDD'] * metrics_df['MDD_norm']
)

### Rank accounts based on performance score


In [16]:
metrics_df['Rank'] = metrics_df['Composite_Score'].rank(ascending=False)


### Extract and save the top 20 best-performing accounts



In [17]:
top_20 = metrics_df.sort_values('Rank').head(20)

metrics_df.to_csv('account_metrics.csv', index=False)
top_20.to_csv('top_20_accounts.csv', index=False)
print(" Analysis complete. Results saved.")


 Analysis complete. Results saved.
